In [ ]:
# General notebook settings
import warnings

warnings.filterwarnings("error", category=DeprecationWarning)

# Time Series Aggregation

In this example, we are going to explore different ways to cluster the temporal resolution of PyPSA models, and what impact they have on the optimisation results and solving times. Using an hourly resolved variant of the [single-node capacity expansion example](), we will compare three different approaches to reduce the number of time steps in the model:

- **Sampling**: Selecting a subset of the given snapshots based on a given frequency,
- **Averaging**: Aggregating the snapshots by averaging them over a given frequency, and
- **Segmentation**: Clustering the snapshots into segments of a given frequency and using [`tsam`](https://tsam.readthedocs.io/en/latest/) library.

We start with the usual imports and loading the hourly resolved model.

In [ ]:
import logging
import time

import pandas as pd

import pypsa

logging.getLogger().setLevel(logging.WARNING)

SOLVER = "highs" # or "gurobi"

template_n = pypsa.Network(
    "https://tubcloud.tu-berlin.de/s/4ra3NKrLGzE42of/download/model-energy-hourly.nc"
)

In [ ]:
template_n

Since we also want to monitor the solving times, we will use a small utility function that wraps around the solving process and returns the seconds it took to solve the model:

In [ ]:
def time_it(func, *args, **kwargs):
    """Time the execution of a function and return the elapsed time in seconds."""
    start_time = time.time()
    result = func(*args, **kwargs)
    elapsed_time = time.time() - start_time
    return result, elapsed_time


logging.getLogger().setLevel(logging.WARNING)

## Hourly Baseline

Additionally, we need a baseline model to compare aggregated models against. We will use the hourly resolved model for this.

In [ ]:
n_hourly = template_n.copy()

_, s_hourly = time_it(n_hourly.optimize, solver_name=SOLVER, log_to_console=False)
s_hourly

## Sampling

We will start with the sampling approach, which is the simplest one. We simply select every $N$-th snapshot from the model. The important part here is that we need to adjust the snapshot weightings accordingly, as each remaining snapshot now represents $N$ hours. We iterate over N from 2 to 11, i.e. from 2-hourly to 11-hourly resolved models.

In [ ]:
sampling_n = {1: n_hourly}
sampling_s = {1: s_hourly}

for resolution in range(2, 12):
    n = template_n.cluster.temporal.downsample(resolution)

    _, s = time_it(n.optimize, solver_name=SOLVER, log_to_console=False)

    sampling_n[resolution] = n
    sampling_s[resolution] = s

## Averaging

The averaging approach also has equal snapshot durations, but instead of selecting data from every $N$-th snapshots, we average time series data for every $N$ snapshots. This means that the resulting model has $N$ times fewer snapshots, but each snapshot represents the average of $N$ original snapshots. Again, we need to adjust the snapshot weightings accordingly.

In [ ]:
averaging_n = {1: n_hourly}
averaging_s = {1: s_hourly}

for resolution in range(2, 12):
    n = template_n.cluster.temporal.resample(f"{resolution}h")

    _, s = time_it(n.optimize, solver_name=SOLVER, log_to_console=False)

    averaging_n[resolution] = n
    averaging_s[resolution] = s

## Segmentation

The segmentation approach is more complex. It uses a separate library called [`tsam`](https://tsam.readthedocs.io/en/latest/) to cluster the snapshots into segments of varying lengths. The sequence of snapshots is preserved as segments are only formed from neighbouring snapshots based on their similarity. For measuring similarity, it is advisable to normalise the time series data. This approach promises to capture the temporal patterns more effectively, as it can opt for higher resolution during periods of high variability and lower resolution during periods with low variability. The snapshot weightings are adjusted based on the number of snapshots in each segment.

In [ ]:
segmentation_n = {1: n_hourly}
segmentation_s = {1: s_hourly}

for resolution in range(2, 12):
    # calculate number of segments equivalent to resolution
    segments = int(8760 / resolution)

    n = template_n.cluster.temporal.segment(segments)

    # run optimization
    _, s = time_it(n.optimize, solver_name=SOLVER, log_to_console=False)

    segmentation_n[resolution] = n
    segmentation_s[resolution] = s

Now before we go ahead with the evaluation of the different approaches, let's quickly glance at the distribution of snapshot durations obtained from the segmentation approach for a resolution equivalent to a 3-hourly model. We can see quite some variability in the segment length.

In [ ]:
segmentation_n[3].snapshot_weightings.generators.value_counts().sort_index(
    ascending=True
).plot.bar(ylabel="snapshots [number]", xlabel="snapshot duration [h]")

## Representative hours

The representative hour (a.k.a., typical hour) approach is yet another approach that you can take which uses the [`tsam`](https://tsam.readthedocs.io/en/latest/) library.
Unlike all the other approaches, it does not preserve the sequence of snapshots over whole time horizon when it aggregates the timeseries.
Instead, it extracts hours(s) which are most representative of the entire timeseries and only keeps those.
The remaining snapshots remain at their original resolution, but there are now only a select number of non-contiguous days in the list of snapshots.

For instance, if aggregating a year into 24 representative hours, you would go from a contiguous timeseries of 8760 hours to a non-contiguous timeseries of 24 snapshots.
Each of these snapshots would represent other hours in the original timeseries and would be weighted accordingly in the objective function.

In [ ]:
typical_hours_n = {8760: n_hourly}
typical_hours_s = {8760: s_hourly}

for resolution in range(2, 12):
    # calculate number of segments equivalent to resolution
    segments = int(8760 / resolution)
    n = template_n.cluster.temporal.representative_hours(
        num_representative_hours=segments,
        clusterMethod="k_means", representationMethod="medoidRepresentation"
    )

    # run optimization
    _, s = time_it(n.optimize, solver_name=SOLVER, log_to_console=False)

    typical_hours_n[segments] = n
    typical_hours_s[segments] = s

Now, let's look at the days that were considered typical.

In [ ]:
import plotly.graph_objs as go


fig = go.Figure()
dropdown_buttons = []

for i, (k, n_) in enumerate(typical_hours_n.items()):
    if n_.storage_snapshots.empty:
        continue
    df_plot = n_.storage_snapshots.set_index("original_snapshot").representative_hour
    df_plot = df_plot.map(df_plot.value_counts())
    fig.add_trace(
        go.Bar(
            x=df_plot.index,
            y=pd.Series(1, index=df_plot.index),
            name=k,
            visible=(i == 0),
            marker={"color": df_plot.values, "colorscale": "viridis"},
        )
    )
    visible = [False] * len(typical_hours_n)
    visible[i] = True
    dropdown_buttons.append(
        {"label": k, "method": "update", "args": [{"visible": visible}]}
    )

# Update layout with dropdown
fig.update_layout(
    updatemenus=[
        {
            "buttons": dropdown_buttons,
            "direction": "down",
            "showactive": True,
            "x": 0,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top",
        }
    ],
    xaxis_title="Snapshot",
    yaxis_title="Typical Period",
)

fig

To capture long-term storage, we apply a specific set of constraints and new auxiliary decision variables (`...<intra|inter>_period...`) when we use typical periods:

In [ ]:
def reconstruct_storage_level(n):
    storage_level = n.components["Store"].dynamic["e"]
    storage_level.index = n.storage_snapshots.original_snapshot.values
    storage_level_reconstructed = storage_level.reindex(n_hourly.snapshots).interpolate()
    return storage_level_reconstructed


ax = (
    n_hourly.components["Store"]
    .dynamic["e"]["hydrogen storage"]
    .plot(
        ylabel="Storage level [MWh]",
        xlabel="snapshot",
        legend=True,
        label="original time series",
    )
)
for n_hours, n_ in typical_hours_n.items():
    if n_hours == 8760:
        continue
    storage_level_reconstructed = reconstruct_storage_level(n_).squeeze()
    storage_level_reconstructed.plot(
        label=f"{n_hours} typical hours", ax=ax, legend=True, alpha=0.7
    )

## Evaluation

Let's start our evaluation with a look at the solving times. We can see that across all approaches, the solving times quickly decay, especially as we go from hourly to 2-hourly resolved models and decrease less substantially afterwards.

In [ ]:
results = {
    "sampling": (sampling_n, sampling_s),
    "averaging": (averaging_n, averaging_s),
    "segmentation": (segmentation_n, segmentation_s),
    "representative hours": (typical_hours_n, typical_hours_s),
}

In [ ]:
def time_series_plot(results):
    for name, (ns, ss) in results.items():
        pd.Series(
            {len(n.snapshots): ss[i] for i, n in ns.items()}, name=name
        ).sort_index().plot(
            ylabel="time [s]", xlabel="N snapshots", legend=True, marker="o"
        )


time_series_plot(results)

Furthermore, we compare the relative error in total system costs compared to the hourly resolved model. We can see how the segmentation approach remains more stable than the other two approaches, especially for lower resolutions. Even with 11-hourly equivalent resolution, the segmentation approach only has a relative error of -1% compared to the hourly resolved model (i.e. it appears to be 1% cheaper). Similar patterns can be observed for the relative error in the total installed capacity of solar and batteries, two technologies that are particularly sensitive to the temporal resolution of the model.

In [ ]:
def error_plot_rel_to_snapshots(results, func, ylabel):
    for name, (ns, ss) in results.items():
        pd.Series(
            {len(n.snapshots): func(n) for n in ns.values()}, name=name
        ).sort_index().div(func(n_hourly)).sub(1).mul(100).plot(
            ylabel=ylabel, xlabel="N snapshots", legend=True, marker="o"
        )


def error_plot_rel_to_solve_time(results, func, ylabel):
    for name, (ns, ss) in results.items():
        pd.Series({ss[k]: func(n) for k, n in ns.items()}, name=name).sort_index().div(
            func(n_hourly)
        ).sub(1).mul(100).plot(
            ylabel=ylabel, xlabel="solve time [s]", legend=True, marker="o"
        )


def tsc(n):
    return (n.statistics.capex().sum() + n.statistics.opex().sum()) / 1e9


error_plot_rel_to_snapshots(results, tsc, "relative objective error [%]")

In [ ]:
def solar(n):
    return n.generators.loc["solar", "p_nom_opt"]


error_plot_rel_to_solve_time(results, solar, "relative solar capacity error [%]")

In [ ]:
def wind(n):
    return n.generators.loc["wind", "p_nom_opt"]


error_plot_rel_to_snapshots(results, wind, "relative wind capacity error [%]")

In [ ]:
def hydrogen(n):
    return n.stores.loc["hydrogen storage", "e_nom_opt"]


error_plot_rel_to_snapshots(results, hydrogen, "relative hydrogen capacity error [%]")

In [ ]:
n.statistics.curtailment(carrier=["wind", "solar"], groupby_time=False).mul(n.storage_snapshots.groupby("representative_hour").weight.sum()).sum()

In [ ]:
def reconstruct_curtailment(n):
    return n.statistics.curtailment(carrier=["wind", "solar"], groupby_time=False).mul(n.storage_snapshots.groupby("representative_hour").weight.sum()).sum()

def curtailment(n):
    if n.has_representative_hours:
        return reconstruct_curtailment(n).sum()
    else:
        return n.statistics.curtailment(carrier=["wind", "solar"]).sum()

error_plot_rel_to_solve_time(results, curtailment, "relative wind&solar curtailment error [%]")

In [ ]:

n_hourly.statistics.curtailment(carrier=["wind", "solar"]).sum()